# コーパスの活用

大きなデータセットを活用した言語モデルの学習を学ぶ。


---

## コーパス

コーパス（corpus）またはテキストコーパスとは、言語学や自然言語処理における大規模なテキストデータの集合を指す。言語モデルの学習にもよく使用される。

hugging faceの[datasets](https://huggingface.co/docs/datasets/index)ライブラリを使うと、様々なコーパスを簡単に取得できる。[日本語wikipediaのコーパス](https://huggingface.co/datasets/llm-book/japanese-wikipedia)を取得してみよう。

In [1]:
import os; os.environ["HF_HUB_DISABLE_XET"] = "1" # なんかこれないとうまくダウンロードできない
from datasets import load_dataset

ds = load_dataset("llm-book/japanese-wikipedia", cache_dir="./cache")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 1363395
    })
})

中身はこんな感じ。

In [2]:
# いくつかのサンプルの冒頭100文字を表示してみる
for i in range(3):
    print(ds["train"][i]["text"][:100])
    print("===")

『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第7話である。プロデューサーでもあるデイヴィッド・
===
ゲオルク（ヨーラン）・ヴァーレンベリ（Georg (Göran) Wahlenberg、1780年10月1日 – 1851年3月22日）は、スウェーデンの博物学者である。カール・ツンベルク（トゥーンベ
===
『進軍』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第8話である。原作小説シリーズ『氷と炎の歌』の作者で脚本家
===


こんなのが100万件以上入っている。

本来、webから取得したデータは非常に汚い。汚いというのは、HTMLタグや広告、重複データなどが含まれているということ。このようなテキストデータをそのまま学習に使用すると、当然モデルの性能は低下するため、前処理によって整える必要がある。しかしその作業は非常に手間がかかるため、本書では前処理済みのデータセットを使用させてもらう。


---

## トークナイザ

Tokenizer

テキストをトークンごとに分割するもの。言語モデルにおいて、モデルが扱う最小単位のことをトークンと呼び、単語や文字、サブワードなどが該当する。

全章では手動で文章を単語ごとに分割していたが、コーパスのような大規模なデータセットを扱う場合、手動でのトークン化は非現実的であるため、自動で分割するシステムが必要である。

文字をトークンとする場合、トークン化は非常に簡単で、そのまま1文字単位で区切ればいい。単語をトークンとする場合、英語の場合はスペースで区切るだけでよく、また日本語の場合は適当な形態素解析器を使う。しかしこれらの手法では文章を効率的に分割できないことが多い。効率的とは、より少ない語彙数で多くの文章を表せること。適切な語彙を設定していない場合、同じ文章を表すのに多くのトークンが必要となり、モデルの学習効率が低下する。

そこで、適切な分割方法をデータセットから学習する手法を用いる。具体的には、データセット内で頻出する文字列を1つのトークン（語彙）としてまとめる。これを指定した語彙数に達するまで繰り返し、効率的な分割方法を獲得する。この場合のトークンはサブワードと呼ばれたりする。

サブワードには基本的に単語よりも細かい分割を行うため、未知語への対応力が高いという利点もある。何ならBPE（Byte Pair Encoding）では文字よりもさらに細かいバイト列での分割を行うため、未知語への対応力はさらに高くなる。

実際にChatGPTで使われているBPEのトークナイザを[こちら](https://platform.openai.com/tokenizer)から試すことができ、入力した文章がどのように分割されているのかを確かめることができる。「鬱」とか「薔薇」とか、日常生活ではあまり使わない漢字を入れてみると「?」のアイコンがいくつか表示される。これは各漢字がバイト列に分割され、1文字を複数のトークンで表現しているということになる。

余談。

言語モデルについて、はじめにこう定義した。

> 言語モデルとは、単語の並びに関する確率モデルである。

この定義に従うなら、サブワードのような単語でないものをトークンとするモデル（現在有名な、おそらく全てのLLM）は言語モデルとは呼べないのかもしれない。

ただまあ、言語モデルというのは単語に限らず自然**言語**処理全般で活用されるモデルなので、単語にこだわりを持つ必要はないんじゃないかな。

hugging faceの[transformers](https://huggingface.co/docs/transformers/index)ライブラリから`AutoTokenizer`を使うと、学習済みのトークナイザを簡単に使用できる。

東北大の日本語BERTのトークナイザを取得してみよう。

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese")

こんな感じでトークン化できる。

In [8]:
tokenizer.tokenize("近年の大規模言語モデルの発展は凄まじいね")

['近年', 'の', '大', '規模', '言語', 'モデル', 'の', '発展', 'は', '凄', '##まじ', '##い', 'ね']

`##`は前のトークンと連結していることを示す特殊な記号。

### 特殊トークン

言語モデルを扱う際、データを扱いやすくするために特殊なトークンを考えることがある。以下に例を示す。

- *Unknown*

未知語を意味するトークン。UNKと略されることが多い。学習データに含まれなかった文字に出会ったときなどに使用する。

- *Begin of Sentence*

文章の先頭を意味するトークン。BOSと略したり、`<s>`などと表記したりする。このトークンを用意することで、モデルに文章の始まりを伝えられるようになる。つまり初のトークンをこちらで指定する必要がなくなる。

- *End of Sentence*

文章の終わりを意味するトークン。EOSと略したり、`</s>`などと表記したりする。このトークンを用意することで、モデルが文章の終わりを伝えられるようになる。これが生成されたら生成を止める、ということ。

学習データの全ての文章の該当箇所（BOSであれば文章の初め、EOSであれば文章の終わり）にこれらのトークンを入れてから学習させることで、そのトークンの意味をモデルは理解する。なおモデルにとってはトークンはただのクラスラベルなので、名前は何でもいい。他のトークンと重複しないように括弧を付けたりすることが多い。`[BOS]`とか`<EOS>`とか。

これらの他にもモデルの用途によって様々な特殊トークンが設定される。実際に先のトークナイザに設定されている特殊トークンはこんな感じ。

In [5]:
tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}


---

## コーパスを活用したマルコフモデルの学習

実際にこれらのコーパスやトークナイザを活用して、マルコフモデルの学習を行ってみよう。

まずは学習データを作成する。先の日本語wikiコーパスを使う。全部は多いので少しだけ。なお今回は特殊トークンの設定は行わない。

In [6]:
ds_mini = ds["train"][:1000]
text = [" ".join(tokenizer.tokenize(t)) for t in ds_mini["text"]]

# samples
for t in text[:3]:
    print(t[:100])
    print("===")

『 勝つ か 死ぬ か 』 は H ##BO ( 日本 で は スター ・ チャンネル が 放送 ) の ファンタジー ・ ドラマ ・ シリーズ で ある 『 ゲーム ・ オブ ##・ ##ス ##ロ
===
ゲオルク ( ヨー ラン ) ・ ヴァー ##レン ##ベリ ( Geor ##g ( G ##ö ##ran ) W ##ah ##le ##n ##berg 、 1780 年 10 月 1 日 –
===
『 進軍 』 は H ##BO ( 日本 で は スター ・ チャンネル が 放送 ) の ファンタジー ・ ドラマ ・ シリーズ で ある 『 ゲーム ・ オブ ##・ ##ス ##ローン ##ズ 
===


これを`markovify`で学習させる。前章でできなかったN=2を試してみる。

In [7]:
import markovify

model = markovify.Text(text, state_size=2)

文章を生成してみる。

In [9]:
for _ in range(3):
    sentence = model.make_sentence()
    print(tokenizer.convert_tokens_to_string(sentence.split()))
    print("===")

トピーカ 市長 は 、 ジブラルタル ・ セカンド ディヴィジョン で 2 位 に なり 、 ジブラルタル の サッカー クラブ 。 歴史 2012 - 13 年 シーズン の ジブラルタル ・ フェニックス FC は 、 アメリカ 合衆 国 カンザス 州 トピーカ 市長 は 、 アメリカ 合衆 国 カンザス 州 トピーカ 市長 は 、 1895 年 に 創設 さ れ 、 1985 年 に 創設 さ れ て いる 。
===
ISO 3166 - 2 : TH この 記事 は 、 ストックホルム 音楽 大学 に 関係 する 人物 の 一覧 で ある 。
===
コートニー ・ コック ヘルマン ・ ベーレンス 音楽 学 者 その 他 イズ ラエル ・ グスマン
===


語彙が増えたことで多様な文章が生成されるようになった。


---

## 深層学習に向けたデータセットの構築

次章以降、深層学習を用いて言語モデルの学習を行っていく。そのためのデータセットを本節で構築しよう。

In [2]:
ds # 再掲

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 1363395
    })
})

### トークナイザの学習

ニューラルネットは扱う語彙数に応じてパラメータ数が変化する。語彙数が多ければ多いほどパラメータ数も増える。学習・推論に使用できる計算資源には制限があるため、適切な語彙数を持つトークナイザが欲しい。

トークナイザを学習させると、指定した数の語彙を持つトークナイザを獲得できる。本節ではそれを行う。

hugging faceの[Tokenizers](https://huggingface.co/docs/tokenizers/index)ライブラリを使用する。トークナイザの学習や保存、読み込みが簡単に行える。

実際に学習させてみよう。Unigramのsentencepiece実装を使用する。

> `tokenizers.SentencePieceUnigramTokenizer`もあるけど、`character_coverage`の設定ができないので、sentencepieceライブラリを使った。これを、指定しないとほとんど使われていないような漢字が大量に語彙に入ってしまう。

In [ ]:
import sentencepiece as spm

# データセットを直接渡すとメモリを圧迫するため、ジェネレータをかませる
def ds_iter():
    for item in ds["train"]["text"]:
        yield item

spm.SentencePieceTrainer.Train(
    sentence_iterator=ds_iter(),
    model_prefix="sp_jawiki",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=0.9995, # どの程度の文字をカバーするか。これより使用頻度の低い文字はUNKになる
    train_extremely_large_corpus=True,
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: sp_jawiki
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_nois

90分ほどで学習が完了した。一旦これをTokenizersの形式に変換する。

In [ ]:
!wget https://raw.githubusercontent.com/google/sentencepiece/master/python/src/sentencepiece/sentencepiece_model_pb2.py

from tokenizers import SentencePieceUnigramTokenizer

tokenizer = SentencePieceUnigramTokenizer.from_spm("sp_jawiki.model")

!rm sentencepiece_model_pb2.py

後処理を設定して、モデルを保存する。

In [4]:
from tokenizers.processors import TemplateProcessing

tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>", # BOS, EOSで囲む処理を指定
    special_tokens=[
        ("<s>", tokenizer.token_to_id("<s>")),
        ("</s>", tokenizer.token_to_id("</s>")),
    ],
)
fpath_tokenizer = "trained/tokenizer-jawiki.json"
tokenizer.save(fpath_tokenizer)

### トークナイズ

学習中にトークナイズを行うのは非効率的なので、事前にトークナイズを行っておく。

In [11]:
from transformers import PreTrainedTokenizerFast

fpath_tokenizer = "trained/tokenizer-jawiki.json"
tokenizer = PreTrainedTokenizerFast(tokenizer_file=fpath_tokenizer)
tokenizer.add_special_tokens({
    "bos_token": "<s>",
    "eos_token": "</s>",
    "pad_token": "<pad>",
});

深層学習では各トークンをIDに変換する。Tokenizersでは`tokenizer.encode()`でIDに変換できる。

In [20]:
text = "人工知能は人間の知能を超えるか？"
ids = tokenizer.encode(text)
ids

[1, 3, 6241, 540, 1245, 11, 1462, 6, 540, 1245, 7147, 98, 1037, 2]

それぞれ次のように対応している。

In [21]:
for id in ids:
    print(f"{id:4} {tokenizer.convert_ids_to_tokens(id)}")

   1 <s>
   3 ▁
6241 人工
 540 知
1245 能
  11 は
1462 人間
   6 の
 540 知
1245 能
7147 を超える
  98 か
1037 ?
   2 </s>


Tokenizerオブジェクトを直接呼び出すと、他に必要ないくつかの情報も一緒に取得できる。

In [22]:
results = tokenizer(text)
results

{'input_ids': [1, 3, 6241, 540, 1245, 11, 1462, 6, 540, 1245, 7147, 98, 1037, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

まあ本書で実装する言語モデルでは`input_ids`以外使わないけど。

では学習データをすべてトークナイズする。

In [ ]:
ds_ids = ds.map(
    lambda examples: tokenizer(examples["text"]),
    batched=True,
    remove_columns=["text"],
)

Map:   0%|          | 0/13634 [00:00<?, ? examples/s]

保存しておく。

In [ ]:
ds_ids.save_to_disk("data/jawiki")

Saving the dataset (0/1 shards):   0%|          | 0/13634 [00:00<?, ? examples/s]

完了。次章以降、これを使って深層学習による言語モデルの学習を行っていく。